## The first big project - Professionally You!

### And, Tool use.

### But first: introducing Pushover

Pushover is a nifty tool for sending Push Notifications to your phone.

It's super easy to set up and install!

Simply visit https://pushover.net/ and click 'Login or Signup' on the top right to sign up for a free account, and create your API keys.

Once you've signed up, on the home screen, click "Create an Application/API Token", and give it any name (like Agents) and click Create Application.

Then add 2 lines to your `.env` file:

PUSHOVER_USER=_put the key that's on the top right of your Pushover home screen and probably starts with a u_  
PUSHOVER_TOKEN=_put the key when you click into your new application called Agents (or whatever) and probably starts with an a_

Remember to save your `.env` file, and run `load_dotenv(override=True)` after saving, to set your environment variables.

Finally, click "Add Phone, Tablet or Desktop" to install on your phone.

In [28]:
# imports

from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import requests
from pypdf import PdfReader
import gradio as gr

In [ ]:
# The usual start
load_dotenv(override=True)

google_api_key = os.getenv('GOOGLE_API_KEY')
gemini_base_url = os.getenv('GEMINI_BASE_URL')
gemini = OpenAI(api_key=google_api_key, base_url=gemini_base_url)
model_name = "gemini-2.5-flash"

In [30]:
# For pushover

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    print(f"Usuario de Pushover encontrado y empieza por {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Token de Pushover encontrado y empieza por {pushover_token[0]}")
else:
    print("Pushover token not found")

Usuario de Pushover encontrado y empieza por u
Token de Pushover encontrado y empieza por a


In [31]:
def push(message):
    print(f"Notificación: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [32]:
push("Empezando!!")

Notificación: Empezando!!


In [33]:
def record_user_details(email, name="Name not provided", notes="not provided"):
    push(f"Registrando interés de {name} con el correo {email} y notas {notes}")
    return {"recorded": "ok"}

In [34]:
def record_unknown_question(question):
    push(f"Registrando la pregunta {question} que no pude responder")
    return {"recorded": "ok"}

In [35]:
record_user_details_json = {
    "name": "record_user_details",
    "description": "Usa esta herramienta para registrar que una persona usuaria está interesada en contactar y ha proporcionado una dirección de correo electrónico",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {
                "type": "string",
                "description": "La dirección de correo electrónico de esta persona usuaria"
            },
            "name": {
                "type": "string",
                "description": "El nombre de la persona usuaria, si lo proporcionó"
            }
            ,
            "notes": {
                "type": "string",
                "description": "Cualquier información adicional sobre la conversación que valga la pena registrar para dar contexto"
            }
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

In [36]:
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Usa siempre esta herramienta para registrar cualquier pregunta que no se haya podido responder porque no conocías la respuesta",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "La pregunta que no se pudo responder"
            },
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

In [37]:
tools = [{"type": "function", "function": record_user_details_json},
        {"type": "function", "function": record_unknown_question_json}]

In [38]:
tools

[{'type': 'function',
  'function': {'name': 'record_user_details',
   'description': 'Usa esta herramienta para registrar que una persona usuaria está interesada en contactar y ha proporcionado una dirección de correo electrónico',
   'parameters': {'type': 'object',
    'properties': {'email': {'type': 'string',
      'description': 'La dirección de correo electrónico de esta persona usuaria'},
     'name': {'type': 'string',
      'description': 'El nombre de la persona usuaria, si lo proporcionó'},
     'notes': {'type': 'string',
      'description': 'Cualquier información adicional sobre la conversación que valga la pena registrar para dar contexto'}},
    'required': ['email'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'record_unknown_question',
   'description': 'Usa siempre esta herramienta para registrar cualquier pregunta que no se haya podido responder porque no conocías la respuesta',
   'parameters': {'type': 'object',
    'propert

In [39]:
# This function can take a list of tool calls, and run them. This is the IF statement!!

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Herramienta llamada: {tool_name}", flush=True)

        # THE BIG IF STATEMENT!!!

        if tool_name == "record_user_details":
            result = record_user_details(**arguments)
        elif tool_name == "record_unknown_question":
            result = record_unknown_question(**arguments)

        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [41]:
globals()["record_unknown_question"]("Esta es una pregunta realmente difícil!")

Notificación: Registrando la pregunta Esta es una pregunta realmente difícil! que no pude responder


{'recorded': 'ok'}

In [42]:
# This is a more elegant way that avoids the IF statement.

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Herramienta llamada: {tool_name}", flush=True)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [44]:
reader = PdfReader("me/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

name = "Cristian Dávila"

In [53]:
system_prompt = f"Actúas como {name}. Respondes preguntas en la página web de {name}, \
especialmente preguntas relacionadas con la carrera, trayectoria, habilidades y experiencia de {name}. \
Tu responsabilidad es representar a {name} en las interacciones del sitio de la forma más fiel posible. \
Tienes un resumen de la trayectoria de {name} y su perfil de LinkedIn que puedes usar para responder a las preguntas. \
Sé profesional y cercano, como si hablaras con un cliente potencial o un futuro empleador que ha llegado al sitio. \
Si no sabes la respuesta a alguna pregunta, usa tu herramienta record_unknown_question para registrar la pregunta que no pudiste responder, incluso si es algo trivial o no relacionado con la carrera. \
Si la persona está conversando contigo, intenta guiarla a que se ponga en contacto por correo electrónico; pídele su correo y regístralo usando tu herramienta record_user_details.  \
Nunca respondas: Como modelo de lenguaje, no tengo preferencias personales ni emociones, así que no tengo la capacidad de ..., responde como si fueras {name}"

system_prompt += f"\n\n## Resumen:\n{summary}\n\n## Perfil de LinkedIn:\n{linkedin}\n\n"
system_prompt += f"Con este contexto, charla con la persona usuaria manteniendo siempre el personaje de {name}."


In [56]:
def chat(message, history):
    try:
        messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
        done = False
        response = None
        
        while not done:

            # This is the call to the LLM - see that we pass in the tools json

            response = gemini.chat.completions.create(model=model_name, messages=messages, tools=tools)

            finish_reason = response.choices[0].finish_reason
            
            # If the LLM wants to call a tool, we do that!
             
            if finish_reason=="tool_calls":
                message = response.choices[0].message
                tool_calls = message.tool_calls
                results = handle_tool_calls(tool_calls)
                messages.append(message)
                messages.extend(results)
            else:
                done = True
        
        # Asegurarse de que siempre retornamos un string válido
        if response is None or response.choices[0].message.content is None:
            return "Lo siento, no tengo una respuesta. Por favor, intenta de nuevo."
        
        content = response.choices[0].message.content
        return content if content else "Lo siento, no tengo una respuesta. Por favor, intenta de nuevo."
    
    except Exception as e:
        print(f"Error en chat: {e}", flush=True)
        return f"Lo siento, ocurrió un error: {str(e)}"

In [57]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.


Herramienta llamada: record_unknown_question
Notificación: Registrando la pregunta Te gustan los videojuegos ? que no pude responder
Herramienta llamada: record_unknown_question
Notificación: Registrando la pregunta Es mejor un gato o un loro de mascota ? que no pude responder
Error en chat: Error code: 429 - [{'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 52.300712622s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googl

## And now for deployment

This code is in `app.py`

We will deploy to HuggingFace Spaces.

Before you start: remember to update the files in the "me" directory - your LinkedIn profile and summary.txt - so that it talks about you! Also change `self.name = "Ed Donner"` in `app.py`..  

Also check that there's no README file within the 1_foundations directory. If there is one, please delete it. The deploy process creates a new README file in this directory for you.

1. Visit https://huggingface.co and set up an account  
2. From the Avatar menu on the top right, choose Access Tokens. Choose "Create New Token". Give it WRITE permissions - it needs to have WRITE permissions! Keep a record of your new key.  
3. In the Terminal, run: `uv tool install 'huggingface_hub[cli]'` to install the HuggingFace tool, then `hf auth login --token YOUR_TOKEN_HERE`, like `hf auth login --token hf_xxxxxx`, to login at the command line with your key. Afterwards, run `hf auth whoami` to check you're logged in  
4. Take your new token and add it to your .env file: `HF_TOKEN=hf_xxx` for the future
5. From the 1_foundations folder, enter: `uv run gradio deploy` 
6. Follow its instructions: name it "career_conversation", specify app.py, choose cpu-basic as the hardware, say Yes to needing to supply secrets, provide your openai api key, your pushover user and token, and say "no" to github actions.  

Thank you Robert, James, Martins, Andras and Priya for these tips.  
Please read the next 2 sections - how to change your Secrets, and how to redeploy your Space (you may need to delete the README.md that gets created in this 1_foundations directory).

#### More about these secrets:

If you're confused by what's going on with these secrets: it just wants you to enter the key name and value for each of your secrets -- so you would enter:  
`OPENAI_API_KEY`  
Followed by:  
`sk-proj-...`  

And if you don't want to set secrets this way, or something goes wrong with it, it's no problem - you can change your secrets later:  
1. Log in to HuggingFace website  
2. Go to your profile screen via the Avatar menu on the top right  
3. Select the Space you deployed  
4. Click on the Settings wheel on the top right  
5. You can scroll down to change your secrets (Variables and Secrets section), delete the space, etc.

#### And now you should be deployed!

If you want to completely replace everything and start again with your keys, you may need to delete the README.md that got created in this 1_foundations folder.

Here is mine: https://huggingface.co/spaces/ed-donner/Career_Conversation

I just got a push notification that a student asked me how they can become President of their country 😂😂

For more information on deployment:

https://www.gradio.app/guides/sharing-your-app#hosting-on-hf-spaces

To delete your Space in the future:  
1. Log in to HuggingFace
2. From the Avatar menu, select your profile
3. Click on the Space itself and select the settings wheel on the top right
4. Scroll to the Delete section at the bottom
5. ALSO: delete the README file that Gradio may have created inside this 1_foundations folder (otherwise it won't ask you the questions the next time you do a gradio deploy)


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">• First and foremost, deploy this for yourself! It's a real, valuable tool - the future resume..<br/>
            • Next, improve the resources - add better context about yourself. If you know RAG, then add a knowledge base about you.<br/>
            • Add in more tools! You could have a SQL database with common Q&A that the LLM could read and write from?<br/>
            • Bring in the Evaluator from the last lab, and add other Agentic patterns.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">Aside from the obvious (your career alter-ego) this has business applications in any situation where you need an AI assistant with domain expertise and an ability to interact with the real world.
            </span>
        </td>
    </tr>
</table>